
# IMU-SMPL Mesh Matching Demo

`Demo.ipynb`의 구도를 유지하되, `IMU ↔ SMPL` 매칭으로 바꾼 데모입니다.

- 상단: 여러 사람 SMPL **mesh** 렌더
- 하단 좌: Query IMU signal
- 하단 우: Query IMU와 각 SMPL 후보의 유사도
- 매 프레임: 예측된 매칭 SMPL만 강조색으로 표시

기본 설정은 `gt_pose` + `wandb/latest-run/files/model_*.pth` 입니다.


In [1]:

import os
import re
import glob
import copy
import random
import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from tqdm import tqdm
import imageio
import inspect
if not hasattr(inspect, "getargspec"):
    inspect.getargspec = inspect.getfullargspec
import smplx

from src.evaluation import matching
from src.models import model_loader, SPITE


In [2]:

# -------------------------
# Config
# -------------------------
SEED = 1337
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Matching/data config
DATASET_SPLIT = "eLIPD"
MODEL_TYPE = "SIE"
SKELETON_SOURCE = "gt_pose"   # 핵심: gt_joint가 아니라 gt_pose
EMBED_DIM = 512
NUM_JOINTS = 24
N_FEATS = 6 if SKELETON_SOURCE == "gt_pose" else 3

# Scene config
N_SUBJECTS = 8
WINDOW_SIZE = 160
SCENE_SEED = 42
FPS = 15

# Mesh render config
MESH_FACE_STRIDE = 8   # 작을수록 디테일 높고 느림
MESH_ALPHA_OTHER = 0.65
MESH_ALPHA_MATCH = 0.95
MESH_COLOR_OTHER = "#bdbdbd"
MESH_COLOR_MATCH = "#d7191c"
MESH_COLOR_QUERY = "#2c7bb6"

# Paths
DATA_PATH_CANDIDATES = [
    "data/LIPD/LIPD_SEQUENCES_256p.pkl",
    "/data/LIPD/LIPD_SEQUENCES_256p.pkl",
]

SMPL_MODEL_CANDIDATES = [
    "/home/kdh/.cache/phalp/3D/models/smpl/SMPL_NEUTRAL.pkl",
    "/home/kdh/.cache/4DHumans/data/smpl/SMPL_NEUTRAL.pkl",
]

OUTPUT_PREFIX = "imu_smpl_mesh_matching"


Device: cuda


In [3]:

def resolve_first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("No existing path among: {}".format(paths))


def resolve_latest_wandb_model(latest_run_dir="wandb/latest-run/files"):
    pattern = os.path.join(latest_run_dir, "model_*.pth")
    ckpts = glob.glob(pattern)
    if not ckpts:
        raise FileNotFoundError("No model_*.pth found under {}".format(latest_run_dir))

    def epoch_of(path):
        m = re.search(r"model_(\d+)\.pth$", os.path.basename(path))
        return int(m.group(1)) if m else -1

    ckpts = sorted(ckpts, key=epoch_of)
    return ckpts[-1]


def make_smpl_model(model_pkl_path, device):
    # smplx expects model_path that contains a subfolder named `smpl`.
    # If a direct file path like .../smpl/SMPL_NEUTRAL.pkl is provided,
    # lift one directory up to .../models.
    model_dir = os.path.dirname(model_pkl_path)
    if os.path.basename(model_dir).lower() == "smpl":
        model_dir = os.path.dirname(model_dir)

    model = smplx.create(
        model_path=model_dir,
        model_type="smpl",
        gender="neutral",
        ext="pkl",
        use_pca=False,
        batch_size=1,
    ).to(device)
    model.eval()
    return model


def pose24x3_to_vertices(smpl_model, pose24x3, device):
    # pose24x3: [24, 3] axis-angle (root + 23 body joints)
    pose = torch.tensor(pose24x3, dtype=torch.float32, device=device).view(24, 3)
    global_orient = pose[0:1, :].reshape(1, 3)
    body_pose = pose[1:, :].reshape(1, -1)  # [1, 69]

    with torch.no_grad():
        out = smpl_model(
            global_orient=global_orient,
            body_pose=body_pose,
            betas=torch.zeros((1, 10), dtype=torch.float32, device=device),
            transl=torch.zeros((1, 3), dtype=torch.float32, device=device),
            return_verts=True,
        )
    verts = out.vertices[0].detach().cpu().numpy()
    return verts


def add_mesh(ax, verts, faces, color, alpha=0.9):
    tris = verts[faces]
    mesh = Poly3DCollection(tris, linewidths=0.02, alpha=alpha)
    mesh.set_facecolor(color)
    mesh.set_edgecolor((0, 0, 0, 0.03))
    ax.add_collection3d(mesh)


In [4]:

# -------------------------
# Resolve paths
# -------------------------
DATA_PATH = resolve_first_existing(DATA_PATH_CANDIDATES)
MODEL_PATH = resolve_latest_wandb_model("wandb/latest-run/files")
SMPL_MODEL_PATH = resolve_first_existing(SMPL_MODEL_CANDIDATES)

print("DATA_PATH:", DATA_PATH)
print("MODEL_PATH (latest-run):", MODEL_PATH)
print("SMPL_MODEL_PATH:", SMPL_MODEL_PATH)


DATA_PATH: data/LIPD/LIPD_SEQUENCES_256p.pkl
MODEL_PATH (latest-run): wandb/latest-run/files/model_145.pth
SMPL_MODEL_PATH: /home/kdh/.cache/phalp/3D/models/smpl/SMPL_NEUTRAL.pkl


In [5]:

# -------------------------
# Load data and model
# -------------------------
with open(DATA_PATH, "rb") as f:
    sequence_datasets = pickle.load(f)
print("Available splits:", list(sequence_datasets.keys()))

model_type_to_modalities = {"S": "SKELETON", "I": "IMU", "P": "PC"}
modalities = [model_type_to_modalities[m].lower() for m in MODEL_TYPE if m not in ["E", "T"]]
print("Modalities:", modalities)

if "skeleton" in modalities:
    if SKELETON_SOURCE == "gt_pose":
        skeleton = model_loader.load_smpl_pose_encoder(EMBED_DIM, NUM_JOINTS, device=device, backbone="transformer")
    else:
        skeleton = model_loader.load_skeleton_encoder(EMBED_DIM, NUM_JOINTS, N_FEATS, device=device)
else:
    skeleton = None

imu = model_loader.load_imu_encoder(EMBED_DIM, device=device) if "imu" in modalities else None
pc = model_loader.load_pst_transformer(EMBED_DIM, device=device) if "pc" in modalities else None
model = SPITE.instantiate_binder(modalities, False, imu, pc, skeleton, None).to(device)

ckpt = torch.load(MODEL_PATH, map_location="cpu")
if isinstance(ckpt, dict):
    for key in ("state_dict", "model_state_dict", "model"):
        if key in ckpt and isinstance(ckpt[key], dict):
            ckpt = ckpt[key]
            break
load_msg = model.load_state_dict(ckpt, strict=False)
print("Checkpoint loaded with strict=False")
print("Missing keys:", len(load_msg.missing_keys), "Unexpected keys:", len(load_msg.unexpected_keys))
model.eval()


Available splits: ['ACCAD', 'BMLmovi', 'LIPD_train', 'AIST', 'CMU', 'eLIPD', 'eTC', 'eDIP']
Modalities: ['skeleton', 'imu']


/home/kdh/despite/.venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Loading class <class 'src.models.SPITE.SIE_BINDER'>
Checkpoint loaded with strict=False
Missing keys: 101 Unexpected keys: 242


SIE_BINDER(
  (imu_encoder): IMUEncoder(
    (lstm): LSTM(48, 512, num_layers=2, batch_first=True)
  )
  (skeleton_encoder): SMPLPoseEncoder(
    (encoder): Encoder_TRANSFORMER(
      (skelEmbedding): Linear(in_features=144, out_features=512, bias=True)
      (sequence_pos_encoder): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (seqTransEncoder): TransformerEncoder(
        (layers): ModuleList(
          (0-7): 8 x TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
            )
            (linear1): Linear(in_features=512, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (linear2): Linear(in_features=1024, out_features=512, bias=True)
            (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True

In [ ]:

# -------------------------
# Encode windows with gt_pose
# -------------------------
encoded_dataset = matching.encode_all(
    copy.deepcopy(sequence_datasets[DATASET_SPLIT]),
    model,
    window_length=24,
    model_type=MODEL_TYPE,
    skeleton_source=SKELETON_SOURCE,
)

print("Encoded subjects:", len(encoded_dataset))
print("Encoded sequences:", sum(len(v) for v in encoded_dataset.values()))


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.08 GiB. GPU 0 has a total capacity of 23.52 GiB of which 250.62 MiB is free. Process 3699167 has 21.81 GiB memory in use. Including non-PyTorch memory, this process has 1.44 GiB memory in use. Of the allocated memory 536.97 MiB is allocated by PyTorch, and 475.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

: 

In [ ]:

# -------------------------
# Build multi-person scene (IMU + SMPL pose windows)
# -------------------------
def sample_scene_imu_smpl(dataset, n_subjects=8, window_size=160, seed=42):
    rng = np.random.default_rng(seed)

    candidates = []
    for subject, seqs in dataset.items():
        for seq, seq_dict in seqs.items():
            if "IMU_EMBS" not in seq_dict or "SKELETON_EMBS" not in seq_dict:
                continue
            if len(seq_dict["IMU_EMBS"]) < window_size or len(seq_dict["SKELETON_EMBS"]) < window_size:
                continue
            candidates.append((subject, seq))

    if not candidates:
        raise ValueError("No candidate sequences with enough length.")

    chosen = rng.choice(len(candidates), size=n_subjects, replace=(len(candidates) < n_subjects))

    scene = {"IMU": [], "SMPL_POSE": [], "IMU_EMBS": [], "SMPL_EMBS": [], "META": []}

    for idx in chosen:
        subject, seq = candidates[int(idx)]
        seq_dict = dataset[subject][seq]
        max_start = len(seq_dict["IMU_EMBS"]) - window_size
        start = int(rng.integers(0, max_start + 1))
        end = start + window_size

        scene["IMU"].append(seq_dict["IMU_W"][start:end])
        scene["SMPL_POSE"].append(seq_dict["SKELETON_W"][start:end])
        scene["IMU_EMBS"].append(seq_dict["IMU_EMBS"][start:end])
        scene["SMPL_EMBS"].append(seq_dict["SKELETON_EMBS"][start:end])
        scene["META"].append((subject, seq, start, end))

    scene["IMU"] = torch.stack(scene["IMU"])
    scene["SMPL_POSE"] = torch.stack(scene["SMPL_POSE"])
    scene["IMU_EMBS"] = torch.stack(scene["IMU_EMBS"])
    scene["SMPL_EMBS"] = torch.stack(scene["SMPL_EMBS"])
    return scene


scene = sample_scene_imu_smpl(encoded_dataset, n_subjects=N_SUBJECTS, window_size=WINDOW_SIZE, seed=SCENE_SEED)

print("IMU:", tuple(scene["IMU"].shape))
print("SMPL_POSE:", tuple(scene["SMPL_POSE"].shape), "(axis-angle)")
print("IMU_EMBS:", tuple(scene["IMU_EMBS"].shape))
print("SMPL_EMBS:", tuple(scene["SMPL_EMBS"].shape))
print("Example meta:", scene["META"][0])


In [ ]:

# -------------------------
# Frame-wise IMU -> SMPL similarity
# sims_f: [T, N_query, N_candidate]
# -------------------------
imu_embs = torch.nn.functional.normalize(scene["IMU_EMBS"].float(), dim=-1)
smpl_embs = torch.nn.functional.normalize(scene["SMPL_EMBS"].float(), dim=-1)

sims_f = torch.einsum("qtd,std->tqs", imu_embs, smpl_embs).cpu().numpy()
pred_idx = sims_f.argmax(axis=2)

print("sims_f:", sims_f.shape)
print("pred_idx:", pred_idx.shape)


In [ ]:

# -------------------------
# Precompute SMPL mesh vertices for scene
# -------------------------
smpl_model = make_smpl_model(SMPL_MODEL_PATH, device)
faces = np.asarray(smpl_model.faces, dtype=np.int32)[::MESH_FACE_STRIDE]

N = scene["SMPL_POSE"].shape[0]
T = scene["SMPL_POSE"].shape[1]

# first frame from each 24-frame window for visualization
# [N, T, 24, 3]
smpl_pose_seq = scene["SMPL_POSE"][:, :, 0].cpu().numpy()

verts_scene = [[None for _ in range(T)] for _ in range(N)]
for s in tqdm(range(N), desc="SMPL mesh precompute: subjects"):
    for t in range(T):
        verts_scene[s][t] = pose24x3_to_vertices(smpl_model, smpl_pose_seq[s, t], device)

print("Precompute done. faces:", faces.shape)


In [ ]:

# -------------------------
# Render query videos (mesh style)
# -------------------------
imus = scene["IMU"][:, :, 0].cpu().numpy()  # [N, T, 48]

scale = 3.6
spacing = 2.0

for query_idx in range(N):
    output_video = f"{OUTPUT_PREFIX}_query_{query_idx}.mp4"
    frames = []

    imu_seq = imus[query_idx]
    imu_plot_ch = list(range(36, 48))

    for t in tqdm(range(T), desc=f"Render Query {query_idx}"):
        query_sims = sims_f[t, query_idx]
        matched_idx = int(pred_idx[t, query_idx])

        fig = plt.figure(figsize=(12, 8))
        gs = GridSpec(2, 2, height_ratios=[2, 1], hspace=0.28)

        # --- Top: multi-person SMPL meshes ---
        ax3d = fig.add_subplot(gs[0, :], projection="3d")

        for i in range(N):
            verts = verts_scene[i][t].copy()
            verts[:, 0] = verts[:, 0] * scale + i * spacing
            verts[:, 1] = verts[:, 1] * scale
            verts[:, 2] = verts[:, 2] * scale

            if i == matched_idx and i == query_idx:
                color = "#7b3294"
                alpha = MESH_ALPHA_MATCH
            elif i == matched_idx:
                color = MESH_COLOR_MATCH
                alpha = MESH_ALPHA_MATCH
            elif i == query_idx:
                color = MESH_COLOR_QUERY
                alpha = 0.85
            else:
                color = MESH_COLOR_OTHER
                alpha = MESH_ALPHA_OTHER

            add_mesh(ax3d, verts, faces, color=color, alpha=alpha)
            ax3d.text(i * spacing, -1.5 * scale, -1.0 * scale, f"S{i+1}", ha="center", fontsize=10)

        total_width = (N + 1) * spacing
        ax3d.set_xlim(-1.0, total_width)
        ax3d.set_ylim(-1.8 * scale, 1.8 * scale)
        ax3d.set_zlim(-1.2 * scale, 1.6 * scale)
        ax3d.set_box_aspect([total_width, 3.6 * scale, 2.8 * scale])
        ax3d.view_init(elev=10, azim=270)
        ax3d.axis("off")
        ax3d.set_title(f"IMU->SMPL Mesh Matching | Query S{query_idx+1} | Pred S{matched_idx+1}", fontsize=13, pad=10)

        # --- Bottom-left: IMU ---
        ax_imu = fig.add_subplot(gs[1, 0])
        for c in imu_plot_ch:
            ax_imu.plot(imu_seq[:t+1, c], linewidth=1.2)
        ax_imu.set_xlim(0, T)
        ax_imu.set_ylim(float(np.min(imu_seq)), float(np.max(imu_seq)))
        ax_imu.set_title(f"Query IMU (S{query_idx+1})", fontsize=12, fontweight="bold")
        ax_imu.tick_params(labelsize=9)

        # --- Bottom-right: similarity bar ---
        ax_bar = fig.add_subplot(gs[1, 1])
        bar_colors = [MESH_COLOR_OTHER] * N
        bar_colors[matched_idx] = MESH_COLOR_MATCH
        if query_idx != matched_idx:
            bar_colors[query_idx] = MESH_COLOR_QUERY
        else:
            bar_colors[query_idx] = "#7b3294"

        bars = ax_bar.bar(np.arange(N), query_sims, color=bar_colors)
        bars[matched_idx].set_edgecolor("black")
        bars[matched_idx].set_linewidth(2)

        ax_bar.set_ylim(0.0, 1.0)
        ax_bar.set_xticks(np.arange(N))
        ax_bar.set_xticklabels([f"S{i+1}" for i in range(N)])
        ax_bar.set_title("Cosine Similarity to Query IMU", fontsize=12, fontweight="bold")
        ax_bar.tick_params(labelsize=9)

        fig.tight_layout()
        fig.canvas.draw()
        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        frames.append(frame)
        plt.close(fig)

    imageio.mimsave(output_video, frames, fps=FPS)
    print("Saved:", output_video)



## Notes

- `SKELETON_SOURCE = "gt_pose"`로 설정되어 있어서, 매칭/시각화 모두 `gt_pose` 기준입니다.
- 체크포인트는 `wandb/latest-run/files/model_*.pth` 중 가장 큰 epoch를 자동 선택합니다.
- 렌더가 느리면 `MESH_FACE_STRIDE`를 더 크게 (`12`, `16`) 바꾸세요.
